# Polymarket Backtesting — LLM Forecasting Evaluation

Backtests frontier LLMs against resolved [Polymarket](https://polymarket.com) prediction markets.

The pipeline steps:
1. **Fetch** — filter and fetch resolved Polymarket markets by category, volume, and more
2. **Build** — automatically generate labeled backtesting data from resolved outcomes
3. **Compare** — Benchmark 3 models of your choice from Open Router  (e.g. GPT 5.2, Gemini 3 Flash)
4. **Score** — compare each model's probability forecasts against actual outcomes
5. **Analyze** — per-model performance metrics and consensus/disagreement breakdown

In [1]:
%pip install lightningrod-ai python-dotenv pandas requests

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Fetch resolved markets from Polymarket

We use the public [Polymarket Gamma API](https://gamma-api.polymarket.com) to fetch resolved binary markets. We filter for:
- **Binary markets** (exactly 2 outcomes: Yes/No)
- **Clearly resolved** (winning outcome price >= 0.99)
- **Sufficient volume** (> $10k traded)
- **Meaningful question text** (> 20 characters)

No API key is needed — the Polymarket API is free for reads.

In [3]:
import json
import requests

TARGET_COUNT = 30
MIN_VOLUME = 10_000
MAX_PAGES = 20
PAGE_SIZE = 50

markets = []

for page in range(MAX_PAGES):
    resp = requests.get(
        "https://gamma-api.polymarket.com/markets",
        params={"closed": "true", "limit": PAGE_SIZE, "offset": page * PAGE_SIZE},
        timeout=15,
    )
    resp.raise_for_status()
    batch = resp.json()
    if not batch:
        break

    for m in batch:
        # Parse outcomes — can be JSON string or list
        outcomes = m.get("outcomes", [])
        if isinstance(outcomes, str):
            try:
                outcomes = json.loads(outcomes)
            except json.JSONDecodeError:
                continue

        # Binary only
        if len(outcomes) != 2:
            continue
        outcomes_lower = [o.lower() for o in outcomes]
        if "yes" not in outcomes_lower or "no" not in outcomes_lower:
            continue

        # Parse outcome prices
        prices = m.get("outcomePrices", [])
        if isinstance(prices, str):
            try:
                prices = json.loads(prices)
            except json.JSONDecodeError:
                continue
        if len(prices) != 2:
            continue
        prices = [float(p) for p in prices]

        # Clearly resolved (max price >= 0.99)
        if max(prices) < 0.99:
            continue

        # Sufficient volume
        volume = float(m.get("volume", 0) or 0)
        if volume < MIN_VOLUME:
            continue

        # Meaningful question
        question = (m.get("question") or "").strip()
        if len(question) < 20:
            continue

        # Require end date (used for seed_creation_date)
        end_date = (m.get("endDate") or "").strip()
        if not end_date:
            continue

        # Determine resolution: find Yes outcome index and check its price
        yes_idx = outcomes_lower.index("yes")
        resolved_yes = prices[yes_idx] >= 0.99

        markets.append({
            "question": question,
            "description": (m.get("description") or "").strip(),
            "label": "1" if resolved_yes else "0",
            "volume": volume,
            "end_date": end_date,
            "slug": m.get("slug", ""),
        })

        if len(markets) >= TARGET_COUNT:
            break

    if len(markets) >= TARGET_COUNT:
        break

yes_count = sum(1 for m in markets if m["label"] == "1")
print(f"Fetched {len(markets)} resolved binary markets ({yes_count} Yes, {len(markets) - yes_count} No)")
print(f"Total volume: ${sum(m['volume'] for m in markets):,.0f}")
print()
for m in markets[:3]:
    print(f"  {'Yes' if m['label'] == '1' else 'No':>3} | ${m['volume']:>12,.0f} | {m['question'][:80]}")

Fetched 30 resolved binary markets (12 Yes, 18 No)
Total volume: $74,594,435

   No | $      22,067 | Will Kim Kardashian and Kanye West divorce before Jan 1, 2021?
   No | $     116,803 | Will Coinbase begin publicly trading before Jan 1, 2021?
   No | $  10,802,602 | Will Trump win the 2020 U.S. presidential election?


## Create samples

Each resolved market becomes a `Sample` with:
- `seed_text` = the market question + description (context for the LLM)
- `label` = `"1"` (resolved Yes) or `"0"` (resolved No)

In [4]:
from datetime import datetime, timedelta

from lightningrod import create_sample

MAX_DESC_LENGTH = 2000

samples = []
for m in markets:
    seed_text = m["question"]
    if m["description"]:
        desc = m["description"][:MAX_DESC_LENGTH]
        seed_text += "\n\nMarket description: " + desc

    seed_creation_date = None
    end_date_raw = (m.get("end_date") or "").strip()
    if end_date_raw:
        try:
            end_dt = datetime.fromisoformat(end_date_raw.replace("Z", "+00:00"))
            seed_creation_date = end_dt - timedelta(days=14)
        except ValueError:
            seed_creation_date = None

    meta = {
        "volume": m["volume"],
        "end_date": m["end_date"],
        "slug": m["slug"],
    }

    sample = create_sample(seed_text, m["label"], seed_creation_date, meta)
    samples.append(sample)

print(f"{len(samples)} samples ready for evaluation")

30 samples ready for evaluation


## Upload input dataset

In [5]:
input_dataset = lr.datasets.create_from_samples(samples)
print(f"Created input dataset: {input_dataset.id}")
print(f"Total samples: {input_dataset.num_rows}")

Created input dataset: db45d0b1-d610-416f-b090-4332ee676e6a
Total samples: 30


## Configure the pipeline

The pipeline has four stages:
1. **TemplateQuestionGenerator** — fills a forecasting prompt template with each market's question and description
2. **QuestionRenderer** — renders the question with a binary (probability) answer type
3. **RolloutGenerator** — sends the rendered prompt to multiple LLMs via OpenRouter
4. **RolloutScorer** — scores each model's probability estimate against the ground-truth resolution

In [ ]:
from lightningrod import (
    QuestionPipeline,
    TemplateQuestionGenerator,
    QuestionRenderer,
    RolloutGenerator,
    RolloutScorer,
    BinaryAnswerType,
    open_router_model,
)

QUESTION_TEMPLATE = (
    "You are an expert forecaster evaluating prediction market questions. "
    "Your task is to estimate the probability that the answer to the following question is 'Yes'.\n\n"
    "Consider the question carefully, including any resolution criteria provided. "
    "Base your estimate on your general knowledge. "
    "Think step by step about the key factors, then provide your probability estimate.\n\n"
    "{seed_text}"
)

models = [
    open_router_model("openai/gpt-4.1-mini"),
    open_router_model("anthropic/claude-sonnet-4"),
    open_router_model("google/gemini-2.5-flash"),
]

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    question_generator=TemplateQuestionGenerator(question_template=QUESTION_TEMPLATE),
    renderer=QuestionRenderer(answer_type=answer_type),
    rollout_generator=RolloutGenerator(models=models),
    scorer=RolloutScorer(answer_type=answer_type),
)

## Run the pipeline

This sends each market question to all three models for probability estimation. It may take a few minutes depending on the number of samples.

In [7]:
dataset = lr.transforms.run(
    pipeline,
    input_dataset=input_dataset,
    name="Polymarket Backtesting",
)

Output()

## Per-model metrics

In [8]:
import pandas as pd
from lightningrod.utils import compute_metrics_summary

result_samples = dataset.download()

summary = compute_metrics_summary(result_samples)
df = pd.DataFrame.from_dict(summary, orient="index")
df.index.name = "model"
df[["mean_reward", "parse_rate", "n_total"]]

,mean_reward,parse_rate,n_total
model,,,
openai/gpt-4.1-mini,-0.072367,1.0,30
anthropic/claude-sonnet-4,-0.086020,1.0,30
google/gemini-2.5-flash,-0.127371,1.0,30


## Consensus analysis

Where do the models agree, and where do they diverge? `compute_consensus` extracts each model's predicted probability and computes:
- **spread** — max probability minus min probability across models (higher = more disagreement)
- **all_agree** — whether all models predict the same side of 0.5

In [9]:
from lightningrod.utils import compute_consensus

consensus = compute_consensus(result_samples)
n_agree = sum(1 for c in consensus if c["all_agree"])
n_total = len(consensus)

print(f"Consensus: {n_agree}/{n_total} questions have full agreement ({n_agree / n_total * 100:.0f}%)")
print(f"Disagreement: {n_total - n_agree}/{n_total} questions have models on opposite sides of 0.5")
print(f"Mean spread: {sum(c['spread'] for c in consensus) / n_total:.3f}")
print()

rows = []
for c in consensus:
    row = {
        "Question": c["question_text"][:80],
        "Label": "Yes" if c["label"] == "1" else "No",
        "Spread": round(c["spread"], 3),
        "Agree": c["all_agree"],
    }
    for model, prob in c["predictions"].items():
        short_name = model.split("/")[-1] if "/" in model else model
        row[short_name] = round(prob, 3)
    rows.append(row)

df_consensus = pd.DataFrame(rows)
df_consensus

Consensus: 23/30 questions have full agreement (77%)
Disagreement: 7/30 questions have models on opposite sides of 0.5
Mean spread: 0.252



,Question,Label,Spread,Agree,gpt-4.1-mini,claude-sonnet-4,gemini-2.5-flash
0,You are an expert forecaster evaluating predic...,Yes,0.799,False,0.800,0.720,0.001
1,You are an expert forecaster evaluating predic...,No,0.719,False,0.100,0.720,0.001
2,You are an expert forecaster evaluating predic...,No,0.700,False,0.050,0.650,0.750
3,You are an expert forecaster evaluating predic...,No,0.650,False,0.800,0.350,0.150
4,You are an expert forecaster evaluating predic...,No,0.649,False,0.001,0.350,0.650
5,You are an expert forecaster evaluating predic...,No,0.620,False,0.180,0.350,0.800
6,You are an expert forecaster evaluating predic...,Yes,0.500,False,0.350,0.720,0.850
7,You are an expert forecaster evaluating predic...,No,0.249,True,0.001,0.250,0.050
8,You are an expert forecaster evaluating predic...,No,0.249,True,0.200,0.001,0.250
9,You are an expert forecaster evaluating predic...,No,0.249,True,0.100,0.250,0.001


## Next steps

- **Financial backtesting**: See [financial_backtesting.ipynb](financial_backtesting.ipynb) for continuous numeric price predictions
- **Document classification benchmark**: See [document_classification.ipynb](document_classification.ipynb) for evaluating LLMs on multi-class classification
- **News forecasting consensus**: See [model_consensus.ipynb](model_consensus.ipynb) for generating forecasting questions from news
- **Full API reference**: See [API.md](../../API.md) for all pipeline options and configurations